In [ ]:
url = "https://www.youtube.com/watch?v=XVJaqRSrEZA"

In [ ]:
import re

re.findall(r"(?:v=)([\w-]{11})", url)

In [ ]:
import re

def extract_youtube_video_id(url):
    # Regular expression to capture the video ID
    pattern = r'(?:v=|\/)([0-9A-Za-z_-]{11}).*'
    match = re.search(pattern, url)
    if match:
        return match.group(1)
    return None


In [ ]:
pattern = r'(?:v=|\/)([0-9A-Za-z_-]{11}).*'
re.search(pattern, "https://www.youtube.com/watch?v=29XE10U6o-oc&t=6s")


In [ ]:
re.search(pattern, "https://www.youtube.com/watch?v=uyiG6uw-6pA&t=230s").group(1)

In [ ]:
extract_youtube_video_id("https://www.youtube.com/watch?v=29XE10U6ooc&t=6s")

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi

# Replace with the YouTube video ID
video_id = "ZPUtA3W-7_I"
# video_id = "giqjHxdN-Ok"
# video_id = "95wUTeGHG1k"

In [ ]:
# ytt_api = YouTubeTranscriptApi()
# transcript_list = ytt_api.fetch(video_id)

In [ ]:
ytt_api = YouTubeTranscriptApi()
transcript_list = ytt_api.get_transcript(video_id, languages=['en','hi'])

In [ ]:
transcript_list[:5]

In [ ]:
type(transcript_list)

In [ ]:
temp_ = ""
for i in transcript_list:
    temp_ += i['text']
len(temp_)

In [ ]:
[True for i in temp_.split(". ") if len(i) > 500]

In [ ]:
68.76+3.719

In [ ]:

chunk_size = 500
chunks = []
timestamps = []

chunk_stack = ""
timestamp_stack = ""

for i in transcript_list:
    the_text = i['text'].strip()
    the_timestamp = i['start']
        
    # if the text is longer than the chunk size
    if len(the_text) >= chunk_size:

        # if previously any chunks exist, append them to the list
        if chunk_stack:
            chunks.append(chunk_stack)
            timestamps.append(timestamp_stack)

            # reset the stack
            chunk_stack = ""
            timestamp_stack = 0

        # append the current chunk
        chunks.append(the_text)
        timestamps.append(the_timestamp)


    # if the text is shorter than the chunk size
    else:
        
        # if chunk and text combined is longer than the chunk size
        if len(chunk_stack := chunk_stack + " " + the_text) > chunk_size:
            
            # split the chunk stack into sentences
            splits = chunk_stack.split(". ")
            temp_chunk_stack = ""
            
            # while the chunk stack is longer than the chunk size
            while(len(chunk_stack := ". ".join(splits)) > chunk_size):
                
                # pop the last sentence from the splits and add it to the temp chunk stack
                temp_chunk_stack = splits.pop() + ". " + temp_chunk_stack
            
            # append the chunk stack and timestamp when the chunk stack is shorter than the chunk size
            chunks.append(chunk_stack.strip())
            timestamps.append(timestamp_stack)

            # reset the stack to temp chunk stack
            chunk_stack = temp_chunk_stack

        # always update the timestamp    
        timestamp_stack = the_timestamp


In [ ]:
len(chunks)

In [ ]:
len(timestamps)

ChromaDB

In [ ]:
import chromadb
chroma_client = chromadb.PersistentClient("./my_chroma_db")

In [ ]:
chroma_client.list_collections()

In [ ]:
# generating embeddings

from google import genai
from dotenv import load_dotenv
import os
load_dotenv()
gemini_api_key = os.getenv("GOOGLE_API_KEY")

client = genai.Client(api_key=gemini_api_key)

class GeminiEmbedding:
    def __call__(self, input):
        result = client.models.embed_content(
            model="models/text-embedding-004",
            contents=input)

        return [e.values for e in result.embeddings]
    def name(self):
        return "gemini_embedding"

In [ ]:
collection = chroma_client.get_or_create_collection("youtube_transcripts",embedding_function=GeminiEmbedding())

In [ ]:
import numpy as np
import uuid

chunks_len = len(chunks)
for i in range(0,chunks_len,100):

    collection.add(
        documents=chunks[i:i+100],
        ids=[str(uuid.uuid4()) for _ in range(100 if chunks_len >= 100 else chunks_len%100)],
        metadatas=[{
            "time_stamp": t,
            "youtube_id": video_id,
        } for t in timestamps[i:i+100] ]
    )
    chunks_len -= 100

In [ ]:
len(collection.get()['ids'])

In [ ]:
iiii = collection.query( query_texts='pollution in India',n_results=10,include=['documents','metadatas'])

In [ ]:
iiii['documents'][0]

In [ ]:
[i['time_stamp'] for i in iiii['metadatas'][0]]

In [ ]:
kkkk = collection.query( query_texts='pollution in India', n_results=5, where={"youtube_id":"PUtA3W-7_I"})

In [ ]:
kkkk

In [ ]:
if kkkk['documents'][0]:
    print("yes")

In [ ]:
collection.get()

In [ ]:
kkkk

In [ ]:
iiii

In [ ]:
client.models.embed_content(
        model="models/text-embedding-004",
        contents='who are you').embeddings[0].values